1 -  Carregar o conteúdo de https://cdn3.gnarususercontent.com.br/4790-python/new/Resenhas_App_ChatGPT.txt, onde cada linha será um elemento da lista do Python.

2 - Mandá-la ao modelo que você está rodando localmente para extrair, em formato JSON, onde cada item terá "usuario", "resenha original", "resenha_pt", "avaliacao" (Positiva, Negativa, Neutra)

3 - Transformar a resposta do modelo em uma lista de dicionários Python

4 - Criar uma função que, dada uma lista de dicionários, percorre a lista e faz 2 coisas:
    a) conta a quantidade de avaliações positivas, negativas e neutras;
    b) une cada item dessa lista em uma variável do tipo string com algum separador.

Ao final, retorna ambas as coisas.

In [6]:
import sys
import json
import re
from urllib.request import urlopen, Request
from openai import OpenAI

# Encoding UTF-8 no Windows (emojis etc.). No Jupyter, stdout não tem reconfigure.
if hasattr(sys.stdout, "reconfigure") and getattr(sys.stdout, "encoding", "") != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8")

# 1 - Carregar conteúdo: cada linha = elemento da lista
# (User-Agent evita 403 Forbidden em alguns CDNs)
url = "https://cdn3.gnarususercontent.com.br/4790-python/new/Resenhas_App_ChatGPT.txt"
req = Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
with urlopen(req) as resp:
    texto = resp.read().decode("utf-8")
linhas = [linha.strip() for linha in texto.strip().split("\n") if linha.strip()]
print(f"Carregadas {len(linhas)} resenhas")

# Cliente da sua IA local (mesma config do chamada-ao-lim.py)
client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="qualquer-coisa")

Carregadas 5 resenhas


In [10]:
# 2 - Enviar ao modelo local e pedir JSON
# Formato das linhas: id$usuario$resenha_original
texto_resenhas = "\n".join(linhas)

prompt = f"""Analise as resenhas abaixo. Cada linha tem o formato: id$nome_usuario$texto_da_resenha.

Retorne um JSON com uma lista de objetos. Cada objeto deve ter exatamente:
- "usuario": nome do usuário
- "resenha_original": texto original da resenha
- "resenha_pt": tradução da resenha para português brasileiro
- "avaliacao": apenas uma destas: "Positiva", "Negativa" ou "Neutra"

Retorne APENAS o JSON (array de objetos), sem texto antes ou depois.

Resenhas:
{texto_resenhas}
"""

response = client.chat.completions.create(
    model="google/gemma-3-4b",
    messages=[
        {"role": "system", "content": "Você extrai dados e retorna somente JSON válido, sem markdown."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
)
resposta_crua = response.choices[0].message.content
print("Resposta do modelo (início):", resposta_crua)

Resposta do modelo (início): ```json
[
  {
    "usuario": "Safoan Riyad",
    "resenha_original": "J'aimais bien ChatGPT. Mais la dernière mise à jour a tout gâché. Elle a tout oublié.",
    "resenha_pt": "Eu gostava muito do ChatGPT. A última atualização arruinou tudo. Ela esqueceu tudo.",
    "avaliacao": "Negativa"
  },
  {
    "usuario": "Habimana Therese",
    "resenha_original": "This app is very important but sometimes it gives lies",
    "resenha_pt": "Este aplicativo é muito importante, mas às vezes ele mente.",
    "avaliacao": "Negativa"
  },
  {
    "usuario": "Shahidatun jannat",
    "resenha_original": "Wonderful app..i just aastonished.. love this app..",
    "resenha_pt": "Ótimo aplicativo... eu fiquei impressionado(a)... amo este aplicativo...",
    "avaliacao": "Positiva"
  },
  {
    "usuario": "Rayyan R",
    "resenha_original": "Anytime i try to get the app to work. It tells me \"an error has occured\" can this be fixed",
    "resenha_pt": "Sempre que tento fazer o

In [11]:
# 3 - Transformar resposta do modelo em lista de dicionários
# (extrair JSON se vier dentro de ```json ... ```)
texto_json = resposta_crua.strip()
match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", texto_json)
if match:
    texto_json = match.group(1).strip()
lista_resenhas = json.loads(texto_json)
print(f"Lista com {len(lista_resenhas)} itens. Exemplo:", lista_resenhas[0] if lista_resenhas else "vazio")

Lista com 5 itens. Exemplo: {'usuario': 'Safoan Riyad', 'resenha_original': "J'aimais bien ChatGPT. Mais la dernière mise à jour a tout gâché. Elle a tout oublié.", 'resenha_pt': 'Eu gostava muito do ChatGPT. A última atualização arruinou tudo. Ela esqueceu tudo.', 'avaliacao': 'Negativa'}


In [12]:
# 4 - Função: contagem de avaliações + unir itens em uma string
def processar_resenhas(lista_dict, separador=" | "):
    """
    Dada uma lista de dicionários com chave "avaliacao":
    a) Conta positivas, negativas e neutras.
    b) Une cada item em uma string com o separador.
    Retorna (contagem, string_unida).
    """
    contagem = {"Positiva": 0, "Negativa": 0, "Neutra": 0}
    partes = []
    for item in lista_dict:
        av = item.get("avaliacao", "").strip().capitalize()
        if av not in contagem:
            contagem[av] = 0
        contagem[av] = contagem.get(av, 0) + 1
        partes.append(json.dumps(item, ensure_ascii=False))
    string_unida = separador.join(partes)
    return contagem, string_unida

contagem, string_unida = processar_resenhas(lista_resenhas)
print("Contagem de avaliações:", contagem)
print("\nPrimeiros 400 caracteres da string unida:", string_unida)

Contagem de avaliações: {'Positiva': 1, 'Negativa': 4, 'Neutra': 0}

Primeiros 400 caracteres da string unida: {"usuario": "Safoan Riyad", "resenha_original": "J'aimais bien ChatGPT. Mais la dernière mise à jour a tout gâché. Elle a tout oublié.", "resenha_pt": "Eu gostava muito do ChatGPT. A última atualização arruinou tudo. Ela esqueceu tudo.", "avaliacao": "Negativa"} | {"usuario": "Habimana Therese", "resenha_original": "This app is very important but sometimes it gives lies", "resenha_pt": "Este aplicativo é muito importante, mas às vezes ele mente.", "avaliacao": "Negativa"} | {"usuario": "Shahidatun jannat", "resenha_original": "Wonderful app..i just aastonished.. love this app..", "resenha_pt": "Ótimo aplicativo... eu fiquei impressionado(a)... amo este aplicativo...", "avaliacao": "Positiva"} | {"usuario": "Rayyan R", "resenha_original": "Anytime i try to get the app to work. It tells me \"an error has occured\" can this be fixed", "resenha_pt": "Sempre que tento fazer o aplic